# Solutions · 15 · Models for nonideal flow

Worked solutions to the exercises of [notebook 15](../notebooks/15_models_for_nonideal_flow.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engreact is missing): install it from GitHub
    import engreact
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engreact"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engreact import rtd
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

A 20 m packed bed with U = 0.5 m/s and D_ax = 0.02 m2/s: compute Pe, sigma^2/tau^2, the equivalent n, and the first-order conversion loss against plug flow for k tau = 3. At what Pe is the loss below 1 %?

In [2]:
U, L, D_ax, k_tau = 0.5, 20.0, 0.02, 3.0
Pe = U * L / D_ax
s2 = rtd.dispersion_variance(Pe)
n_eq = 1 / s2
X_pfr = 1 - np.exp(-k_tau)
X_disp = rtd.dispersion_conversion(k_tau / 1.0, 1.0, Pe)
print(f"Pe = {Pe:.0f}, sigma^2/tau^2 = {s2:.4f}, equivalent n = {n_eq:.0f}; conversion {X_disp:.4f} vs plug flow {X_pfr:.4f} (loss {X_pfr - X_disp:.4f})")
Pe_1pct = next(p for p in np.logspace(0, 4, 400) if (X_pfr - rtd.dispersion_conversion(k_tau, 1.0, p)) / X_pfr < 0.01)
print(f"the loss falls below 1 % of the conversion for Pe above about {Pe_1pct:.0f}")

Pe = 500, sigma^2/tau^2 = 0.0040, equivalent n = 251; conversion 0.9493 vs plug flow 0.9502 (loss 0.0009)
the loss falls below 1 % of the conversion for Pe above about 45


A long packed bed has a Peclet number in the hundreds and behaves as plug flow to within a tenth of a
percent; dispersion matters in short beds, laminar tubes and vessels with internals. For this conversion the
loss drops below 1 % above Pe of about 45; the common rule of thumb of Pe > 100 carries a margin.

## Exercise 2

For a half-order reaction (concave rate), which of segregation and maximum mixedness gives the higher conversion? Test with the CSTR RTD and explain with the shape of the rate law.

In [3]:
t = np.linspace(0, 300, 12001)
E_c = rtd.cstr_E(t, 10.0)
k_half, C_A0 = 0.05, 1.0
seg = rtd.segregation_conversion(t, E_c, k_half, 0.5, C_A0)
mm = rtd.maximum_mixedness_conversion(t, E_c, k_half, 0.5, C_A0)
print(f"half-order reaction, CSTR RTD: segregation {seg:.4f}, maximum mixedness {mm:.4f} -> early mixing gives MORE conversion for a concave rate law")
seg2, mm2 = rtd.segregation_conversion(t, E_c, 0.05, 2, 1.0), rtd.maximum_mixedness_conversion(t, E_c, 0.05, 2, 1.0)
print(f"second order for comparison: segregation {seg2:.4f}, maximum mixedness {mm2:.4f}")

half-order reaction, CSTR RTD: segregation 0.3750, maximum mixedness 0.3903 -> early mixing gives MORE conversion for a concave rate law
second order for comparison: segregation 0.2773, maximum mixedness 0.2679


For a rate law concave in concentration (order below one), averaging concentrations before reacting is
better than reacting separately - Jensen's inequality in reverse - so maximum mixedness beats segregation.
The convex second-order case goes the other way. The sign of the difference is the curvature of the rate law.

## Exercise 3

**Implement it yourself:** compute the conversion for n tanks in series analytically for first order, and by feeding the tanks-in-series E(t) into the segregation model; show they agree for n = 1, 2, 5.

In [4]:
k, tau = 0.1, 10.0
t = np.linspace(0, 400, 40001)
for n in (1, 2, 5):
    analytic = 1 - (1 + k * tau / n) ** (-n)
    numeric = rtd.segregation_conversion(t, rtd.tanks_in_series_E(t, tau, n), k)
    print(f"n = {n}: analytic {analytic:.5f}, segregation on the tanks RTD {numeric:.5f}")

n = 1: analytic 0.50000, segregation on the tanks RTD 0.50000
n = 2: analytic 0.55556, segregation on the tanks RTD 0.55556
n = 5: analytic 0.59812, segregation on the tanks RTD 0.59812


For first order the segregation integral over the tanks-in-series E(t) reproduces the closed form exactly -
the RTD suffices, and any consistent route gives the same answer.